<a href="https://colab.research.google.com/github/sara-sgit/Plant-Disease-Classification/blob/main/__VGG16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from keras.models import Sequential, load_model
from keras.layers import Dense, Dropout, Flatten , BatchNormalization
from keras.layers.convolutional import Conv2D,AveragePooling2D, MaxPooling2D
from keras.utils.np_utils import to_categorical
from keras import callbacks # to use the early stopping
from keras.callbacks import ModelCheckpoint  #to save the model with the best accuracy


import tensorflow as tf
import pickle as pk
import numpy as np


from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from keras.models import Model
from sklearn.metrics import classification_report
from google.colab import drive
drive.mount('/content/drive')
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
from tensorflow.keras.layers import Input, Lambda, Dense, Flatten
from tensorflow.keras.models import Model
from keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator,load_img
from tensorflow.keras.models import Sequential
import numpy as np
from glob import glob

# Use the Image Data Generator to import the images from the dataset
from tensorflow.keras.preprocessing.image import ImageDataGenerator


Mounted at /content/drive


In [ ]:

# Check TensorFlow version
tf.__version__

'2.12.0'

In [ ]:

# Configure GPU settings to manage memory usage
config = ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.85 #you can try 0.95
config.gpu_options.allow_growth = True
session = InteractiveSession(config=config)


In [ ]:


# Define image size and dataset paths
IMAGE_SIZE = [224, 224]

train_path = '/content/drive/MyDrive/data2/train'
valid_path = '/content/drive/MyDrive/data2/test'



# Load the VGG16 base model
# Here we will be using imagenet weights
vgg_16 = VGG16(input_shape=IMAGE_SIZE + [3], weights='imagenet', include_top=False)    #include_top=False, the fully connected layers are excluded from the imported model

# don't train existing weights
# Freeze VGG16 layers to prevent retraining
for layer in vgg_16.layers:
    layer.trainable = False

58889256/58889256 [==============================] - 2s 0us/step


In [ ]:

# Initialize data generators for training and validation sets
from keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0,
                                   zoom_range = 0,
                                   horizontal_flip = False)

test_datagen = ImageDataGenerator(rescale = 1./255,shear_range = 0,
                                   zoom_range = 0,
                                   horizontal_flip = False)



# Create the data generators
# Make sure you provide the same target size as initialied for the image size
training_set = train_datagen.flow_from_directory(train_path,
                                                 target_size = (224, 224),
                                                 batch_size = 16,
                                                 class_mode = 'categorical')


test_set = test_datagen.flow_from_directory(valid_path,
                                            target_size = (224, 224),
                                            batch_size = 16,
                                            shuffle=False,
                                            class_mode = 'categorical')

Found 12811 images belonging to 10 classes.
Found 3200 images belonging to 10 classes.


In [ ]:
vgg_16.__dict__

In [ ]:

  # Add custom layers on top of VGG16
folders = glob(train_path+'/*')
# our layers - you can add more if you want
x = Flatten()(vgg_16.output)
x = Dense(4096, activation='sigmoid', name='fc1')(x) # the layer that we gonna take
x = Dense(10, activation='softmax', name='predictions')(x)



In [ ]:
folders


In [ ]:
#prediction = Dense(len(folders), activation='softmax')(x)

# Define the final model
model = Model(inputs=vgg_16.input, outputs=x)

model.summary()

model.compile(
  loss='categorical_crossentropy',
  optimizer='adam',
  metrics=['accuracy']
)

In [ ]:
# Set up callbacks for early stopping and saving the best model
from keras.callbacks import ModelCheckpoint
EPOCH=100
path='/content/drive/MyDrive/Models/Vgg16sigmoid_'+str(EPOCH)+'e.h5'
es = callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=50,restore_best_weights = True)
mc = ModelCheckpoint(path, monitor='val_accuracy', mode='max', save_best_only=True,verbose= True)

In [ ]:
#train the model
# Run the cell. It will take some time to execute
r = model.fit(
  training_set,
  validation_data=test_set,
  epochs=EPOCH,
  callbacks=[es, mc],
  steps_per_epoch=len(training_set),
  validation_steps=len(test_set)
)

In [ ]:
# Load the best model
model=load_model(path)

In [ ]:
# Make predictions and evaluate performance
y_predrob = model.predict(test_set)
from sklearn.metrics import classification_report, confusion_matrix

import numpy as np
y_pred = np.argmax(y_predrob, axis=1)
print('confusion matrix')
print(confusion_matrix(test_set.classes,y_pred))
labels=test_set.labels
sum((labels==y_pred)*1)/len(y_pred)


200/200 [==============================] - 15s 75ms/step
confusion matrix
[[410   9   1   0   2   0   0   3   0   0]
 [  4 143  26   1  11   6   6   2   1   0]
 [  1  13 350   3   8   0   4   3   0   0]
 [  1   9   9 158   4   2   4   1   2   0]
 [  6   8   7   2 318   3   5   0   5   0]
 [  0   1   2   1   1 310  14   0   3   1]
 [  0   0   4   2   8  22 238   0   0   7]
 [  3   6   1   0   1   5   0 626   0   0]
 [  0   1   0   1   3   0   0   0  70   0]
 [  0   2   0   0   1   2  10   0   0 303]]


0.914375

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix



# Calcul de la matrice de confusion
conf_matrix = confusion_matrix(labels, y_pred)

# Calcul des pourcentages dans la matrice de confusion
conf_matrix_percentage = conf_matrix / np.sum(conf_matrix, axis=1, keepdims=True) * 100

# Création de l'étiquette des classes
labels = np.unique(labels)
labels = [str(label) for label in labels]

# Affichage de la matrice de confusion avec des pourcentages sans le signe "%"
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_percentage, annot=True, fmt='.2f', cmap='Blues', cbar=True, square=True,
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Classe prédite')
plt.ylabel('Classe réelle')
plt.title('Matrice de confusion en pourcentage')
plt.show()

In [ ]:
# Feature extraction for further analysis
train_set1 = train_datagen.flow_from_directory(valid_path,
                                            target_size = (200, 200),
                                            batch_size = 16,
                                            shuffle=False,
                                            class_mode = 'categorical')

In [ ]:
  output1 = model.get_layer('fc1').output
  new_model = tf.keras.models.Model(inputs=model.input, outputs=output1)
  O1 = new_model.predict(train_set1)
  print(O1)
  print(O1.shape)

In [ ]:
  output2 = model.get_layer('fc1').output
  new_model = tf.keras.models.Model(inputs=model.input, outputs=output2)
  O2 = new_model.predict(test_set)
  print(O2)
  print(O2.shape)

200/200 [==============================] - 14s 70ms/step
[[1.00000000e+00 1.17384256e-29 1.00000000e+00 ... 1.00000000e+00
  4.36202859e-14 0.00000000e+00]
 [1.00000000e+00 2.81704232e-28 1.00000000e+00 ... 1.00000000e+00
  3.34865091e-14 0.00000000e+00]
 [1.00000000e+00 5.04982856e-30 1.00000000e+00 ... 1.00000000e+00
  1.07149016e-14 0.00000000e+00]
 ...
 [1.00000000e+00 4.65801998e-28 1.00000000e+00 ... 1.00000000e+00
  1.25079016e-12 0.00000000e+00]
 [1.00000000e+00 8.29375615e-27 1.00000000e+00 ... 1.00000000e+00
  9.95094942e-16 0.00000000e+00]
 [1.00000000e+00 1.14405651e-27 1.00000000e+00 ... 1.00000000e+00
  4.14295063e-14 0.00000000e+00]]
(3200, 4096)


In [ ]:
# Save extracted features using pickle
pk.dump(O2,open('/content/drive/MyDrive/features_test_pickle_sigmoid80', 'wb'))

# Quick check of feature shape and values

In [ ]:
ss=O2[3100]
print(ss.shape)
print(max(ss))

(4096,)
56.15979


In [ ]:
O2.shape

(3200, 4096)